## 🎯 Learning Objectives
* Understand the fundamental limitations of naive Retrieval Augmented Generation (RAG) systems.
* Identify common failure modes of naive RAG, such as hallucinations, outdated information, and poor complex query handling.
* Grasp the core principles of how agentic approaches address these limitations through reasoning, planning, and tool use.
* Differentiate between the capabilities of a static RAG pipeline and a dynamic, agent-driven RAG system.


## The Glass Ceiling of Naive RAG: Why Agents Are the Next Frontier

In the rapidly evolving landscape of AI, Retrieval Augmented Generation (RAG) has emerged as a powerful technique to ground Large Language Models (LLMs) in factual, up-to-date information. By retrieving relevant documents from a knowledge base and feeding them to the LLM as context, RAG significantly reduces hallucinations and improves the accuracy of responses. However, as we push the boundaries of what LLMs can do, the limitations of a "naive" RAG approach become increasingly apparent.

### What is Naive RAG?

Imagine a highly efficient librarian whose only instruction is: "When a patron asks a question, find books with keywords matching the question, hand them to the patron, and let them figure it out." This is akin to naive RAG. It typically involves:

1.  **Indexing**: Chunking a document corpus and embedding these chunks into a vector database.
2.  **Retrieval**: When a query comes in, embedding the query and finding the top-k most similar document chunks from the vector database.
3.  **Augmentation**: Prepending these retrieved chunks to the user's query as context.
4.  **Generation**: Passing the augmented prompt to an LLM to generate a response.

### The Cracks in the Foundation: Limitations of Naive RAG

While effective for many use cases, this straightforward approach hits several roadblocks when faced with real-world complexity:

1.  **Fixed Retrieval Strategy**: Naive RAG uses a single, predetermined retrieval method (e.g., semantic similarity). It can't adapt if the query requires a different approach, like keyword search, graph traversal, or a specific API call.
2.  **Lack of Reasoning over Retrieved Context**: The LLM receives the retrieved chunks but doesn't actively *reason* about their relevance or sufficiency *before* generating a response. It simply consumes them. If the retrieved context is irrelevant, contradictory, or incomplete, the LLM might still hallucinate or provide a poor answer.
3.  **Sensitivity to Chunking and Embedding Quality**: The performance is heavily dependent on how documents are chunked and the quality of the embeddings. Suboptimal chunking can lead to missing crucial context or retrieving too much noise.
4.  **Inability to Handle Multi-Step or Complex Queries**: Questions requiring sequential information gathering, logical deduction across multiple sources, or external tool usage (e.g., performing a calculation, looking up live data) are beyond naive RAG's capabilities.
5.  **Hallucinations Persist**: While reduced, hallucinations can still occur if the retrieved context is misleading, insufficient, or if the LLM prioritizes its parametric knowledge over the provided context.
6.  **Outdated Information**: If the knowledge base isn't updated frequently, the RAG system will still provide outdated information, as it has no mechanism to verify freshness.

### The Agentic Leap: How Agents Solve These Problems

Now, imagine our librarian is not just efficient but also intelligent and proactive. When a patron asks a question, this **agentic librarian** can:

*   **Understand the Intent**: "This question isn't just about keywords; it needs a specific type of data." (Planning & Reasoning)
*   **Choose the Right Tool**: "I need to check the latest financial reports, not just general history books." (Tool Use)
*   **Ask Clarifying Questions**: "Are you interested in the current year's data or historical trends?" (Self-Correction & Dialogue)
*   **Perform Multi-Step Searches**: "First, I'll find the company's annual report, then I'll look for the specific section on revenue growth, and finally, I'll cross-reference it with market news." (Multi-step Planning)
*   **Synthesize and Verify**: "These two sources seem to contradict each other; I should find a third source to confirm." (Self-Correction & Verification)

This is the essence of **Agentic RAG**. Agents, powered by LLMs, introduce a layer of intelligence, planning, and tool-use capabilities that transform RAG from a static retrieval mechanism into a dynamic, adaptive problem-solving system. They can:

*   **Dynamically Select Retrieval Strategies**: Choose between semantic search, keyword search, graph queries, or even external APIs based on query intent.
*   **Reason Over Retrieved Information**: Evaluate the relevance and sufficiency of retrieved documents, perform follow-up searches, or rephrase queries.
*   **Utilize External Tools**: Integrate with databases, APIs, calculators, web search engines, or even other LLMs to gather information beyond their internal knowledge base.
*   **Engage in Multi-Step Reasoning**: Break down complex problems into smaller, manageable sub-problems, execute them sequentially, and synthesize the results.
*   **Self-Correct and Iterate**: Identify when a generated answer is likely incorrect or incomplete and initiate a corrective action, such as re-retrieving, using a different tool, or asking for clarification.

By embedding these agentic capabilities, we move beyond simply augmenting an LLM with context to creating a system that can intelligently *navigate* and *interact* with information, leading to significantly more robust, accurate, and versatile RAG applications. This is where frameworks like LangGraph shine, enabling the orchestration of these complex, multi-step agentic workflows.


In [ ]:
# This cell demonstrates a naive RAG approach and highlights its limitations.
# We'll simulate a scenario where naive RAG struggles with a complex or dynamic query.

# Ensure you have the necessary libraries installed:
# pip install langchain-community langchain-core langchain-openai chromadb beautifulsoup4

import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- Configuration --- 
# Set your OpenAI API key. For local LLMs, you might use Ollama or another provider.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Using a placeholder for OpenAI or a local LLM via Ollama for demonstration.
# If using OpenAI, uncomment the line above and replace with your key.
# If using Ollama, ensure it's running and you have a model pulled (e.g., 'llama3').
llm_model = "llama3" # Example for Ollama. Use "gpt-4o" for OpenAI.

# --- 1. Simulate a Knowledge Base (Naive RAG Corpus) ---
# We'll load a simple web page as our knowledge base.
# This knowledge base will be static and won't contain real-time or complex data.

print("--- Setting up Naive RAG Knowledge Base ---")
loader = WebBaseLoader("https://www.agenticlabs.ng/about") # A simple, static page
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs = text_splitter.split_documents(docs)

# Using an in-memory ChromaDB for simplicity. For production, use persistent storage.
# For embeddings, use OpenAIEmbeddings or a local alternative like HuggingFaceEmbeddings.
embeddings = OpenAIEmbeddings() # Requires OPENAI_API_KEY
# For local embeddings, you could use: from langchain_community.embeddings import OllamaEmbeddings; embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(documents=splitted_docs, embedding=embeddings)
retriever = vectorstore.as_retriever()

print(f"Knowledge base loaded with {len(splitted_docs)} chunks.")

# --- 2. Define the Naive RAG Chain ---
# This chain simply retrieves documents and passes them to the LLM.

# Initialize the LLM. Adjust model name based on your setup (OpenAI or Ollama).
if "OPENAI_API_KEY" in os.environ and llm_model.startswith("gpt"): # Check if OpenAI key is set and model is GPT
    llm = ChatOpenAI(model=llm_model, temperature=0)
else:
    # Fallback to Ollama if OpenAI key not set or using a non-GPT model name
    from langchain_community.chat_models import ChatOllama
    llm = ChatOllama(model=llm_model, temperature=0)
    print(f"Using local LLM: {llm_model} via Ollama.")


rag_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant for question-answering tasks. Use the following retrieved context to answer the question. If you don't know the answer, just say that you don't know. Keep the answer concise."),
    ("human", "Context: {context}\nQuestion: {question}")
])

naive_rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt_template
    | llm
    | StrOutputParser()
)

print("--- Naive RAG Chain Ready ---")

# --- 3. Demonstrate Naive RAG Limitations ---

# Limitation 1: Inability to answer questions requiring real-time data or external tools.
# Our knowledge base (agenticlabs.ng/about) does not contain current stock prices.
query_1 = "What is the current stock price of NVIDIA?"
print(f"\nQuery 1 (Naive RAG): {query_1}")
response_1 = naive_rag_chain.invoke(query_1)
print(f"Response 1: {response_1}")
# Expected: The LLM will likely state it doesn't know or hallucinate, as the context won't contain this info.

# Limitation 2: Inability to perform multi-step reasoning or calculations not explicitly in text.
# Our knowledge base might mention 'AI' and 'automation' but not provide a direct calculation.
query_2 = "If AgenticLabs.ng's revenue from AI solutions doubled last year and automation tools grew by 50%, what was their total growth rate?"
print(f"\nQuery 2 (Naive RAG): {query_2}")
response_2 = naive_rag_chain.invoke(query_2)
print(f"Response 2: {response_2}")
# Expected: The LLM will struggle to perform this calculation or synthesize hypothetical data, even if it finds relevant terms.

# Limitation 3: Sensitivity to context relevance for complex, nuanced questions.
# Our knowledge base is about AgenticLabs.ng. A question about a competitor might get confused.
query_3 = "What are the core offerings of DeepMind, and how do they compare to AgenticLabs.ng's mission?"
print(f"\nQuery 3 (Naive RAG): {query_3}")
response_3 = naive_rag_chain.invoke(query_3)
print(f"Response 3: {response_3}")
# Expected: The LLM might focus too much on AgenticLabs.ng or struggle to provide a balanced comparison without external context on DeepMind.

# --- Conceptualizing the Agentic Solution (No full code here, as it's a concept lesson) ---
# An agentic system would address these limitations by:

# For Query 1 (NVIDIA stock price):
# An agent would recognize "stock price" as requiring a financial data tool.
# It would call a `StockPriceTool(company_name="NVIDIA")`.
# The tool would query a real-time financial API and return the current price.
# The agent would then use this real-time data to answer the question.

# For Query 2 (Revenue growth calculation):
# An agent would identify the need for calculation and potentially data retrieval.
# It might first use a `SearchTool(query="AgenticLabs.ng revenue AI automation")` to find relevant financial statements.
# If data is found, it would then use a `CalculatorTool(expression="(AI_revenue * 2 + Automation_revenue * 1.5) / (AI_revenue + Automation_revenue) - 1")`.
# If data is not found, it might state it needs more information or use a `HypotheticalDataGeneratorTool` if allowed.

# For Query 3 (DeepMind comparison):
# An agent would use a `MultiSourceSearchTool` or `WebSearchTool(query="DeepMind core offerings")` to gather information on DeepMind.
# It would then retrieve internal documents on AgenticLabs.ng's mission.
# Finally, it would use its reasoning capabilities to synthesize a comparative analysis based on both external and internal data.

# Cleanup (optional, if you want to remove the ChromaDB directory)
# vectorstore.delete_collection()
# print("ChromaDB collection deleted.")


### Interpreting the Naive RAG Output and the Agentic Advantage

When you run the code above, you'll likely observe the following for the naive RAG responses:

*   **Query 1 (NVIDIA Stock Price)**: The LLM, constrained by the static context from `agenticlabs.ng/about`, will almost certainly state that it cannot find the information or will provide a generic, unhelpful answer. It has no mechanism to access real-time financial data.
*   **Query 2 (Revenue Growth Calculation)**: The LLM might try to infer or even hallucinate numbers, or it will admit it lacks the specific data to perform the calculation. It cannot perform complex arithmetic or synthesize hypothetical scenarios without explicit instructions and data.
*   **Query 3 (DeepMind Comparison)**: The LLM will likely struggle to provide a comprehensive and balanced comparison. It might over-emphasize AgenticLabs.ng (because that's its primary context) or provide a very high-level, generic answer about DeepMind if its parametric knowledge is limited or not prioritized.

These outputs clearly illustrate the "glass ceiling" of naive RAG. It's excellent for questions directly answerable by its pre-indexed knowledge base but falls short when queries demand:

*   **Real-time information**: Data that changes frequently or isn't part of the static corpus.
*   **Complex reasoning**: Multi-step logic, calculations, or synthesis across disparate pieces of information.
*   **External tool interaction**: The need to use APIs, databases, or web search beyond simple document retrieval.
*   **Dynamic adaptation**: Changing the retrieval strategy based on the query's nature.

### Performance Trade-offs and Typical Use Cases

While agentic RAG offers significant advantages, it's crucial to understand the trade-offs:

*   **Increased Latency**: Agents involve multiple steps (planning, tool calls, re-evaluation), which inherently adds latency compared to a single-pass naive RAG system.
*   **Higher Cost**: More LLM calls (for planning, tool use, reflection) and potentially more complex infrastructure lead to higher operational costs.
*   **Increased Complexity**: Designing, debugging, and maintaining agentic workflows (especially with frameworks like LangGraph) is more complex than a linear RAG chain.

**Typical Use Cases for Agentic RAG:**

Despite the trade-offs, agentic RAG is indispensable for scenarios demanding high accuracy, robustness, and dynamic capabilities:

1.  **Complex Question Answering**: Queries requiring information from multiple sources, logical deduction, or synthesis (e.g., "Compare the Q3 earnings of company X and Y, and predict their stock movement based on recent market news.").
2.  **Self-Correction and Verification**: Systems that need to identify potential errors or hallucinations and automatically seek corrective information (e.g., "Is this fact true? If not, find the correct information.").
3.  **Dynamic Information Retrieval**: When the type of information needed varies greatly, requiring different retrieval methods or external APIs (e.g., "Find the latest research papers on quantum computing, then summarize the key findings, and check for any related patents.").
4.  **Automated Research and Analysis**: Agents can perform multi-step research tasks, gather data from various web sources, and generate reports.
5.  **Interactive Problem Solving**: Chatbots that can engage in a dialogue, ask clarifying questions, and use tools to solve user problems iteratively.

In essence, naive RAG is a powerful hammer for nails, but agentic RAG provides a full toolbox, allowing for more intricate and adaptive construction. The choice depends on the complexity and dynamic nature of the problem you're trying to solve.


### Resources for Further Exploration

*   **LangGraph Documentation**: The official hub for building robust, stateful multi-actor applications with LLMs. This is the core framework for agentic RAG.
    *   [LangGraph Official Docs](https://langchain-ai.github.io/langgraph/)
    *   [LangGraph Tutorials](https://langchain-ai.github.io/langgraph/tutorials/)

*   **Research Papers on Agentic RAG Concepts**:
    *   **Self-RAG**: [What Is Self-RAG?](https://www.anyscale.com/blog/what-is-self-rag-llm-retrieval-augmentation)
    *   **Corrective RAG (CRAG)**: [Corrective RAG: Learning to Generate Factually Consistent Text](https://arxiv.org/abs/2401.15884)
    *   **RAG-Fusion**: [RAG-Fusion: A New Take on Retrieval Augmented Generation](https://towardsdatascience.com/rag-fusion-a-new-take-on-retrieval-augmented-generation-511b19f7535)

*   **LLM Providers & Platforms (2026 Ecosystem)**:
    *   **Google AI Studio / Gemini API**: [Google AI Studio](https://ai.google.dev/)
    *   **OpenAI API**: [OpenAI Platform](https://platform.openai.com/)
    *   **Anthropic Claude**: [Anthropic Console](https://console.anthropic.com/)
    *   **Ollama**: Run open-source LLMs locally. [Ollama Website](https://ollama.com/)

*   **Vector Database Solutions**:
    *   **ChromaDB**: An open-source, embeddable vector database. [ChromaDB Docs](https://docs.trychroma.com/)
    *   **Pinecone**: A managed vector database for large-scale applications. [Pinecone Docs](https://www.pinecone.io/docs/)
    *   **Weaviate**: An open-source vector database with semantic search capabilities. [Weaviate Docs](https://weaviate.io/developers/weaviate/)

*   **LangChain Framework**: The foundational library for building LLM applications, often used with LangGraph.
    *   [LangChain Documentation](https://python.langchain.com/docs/get_started/introduction)
